# Two Genie spaces, one question box: an LLM routes questions through MCP

You have **two Genie spaces** already. This notebook connects to each one as an **MCP server**
(Databricks hosts one for every Genie space). It then lets an **LLM decide which Genie space to ask** for each question.

```text
                                  tools/list, tools/call (JSON-RPC over HTTPS)
                              +------------------------------------------------+
  your question               |                                                v
       |            +---------+---------+         MCP server 1: /api/2.0/mcp/genie/<space 1>  --> Genie space 1
       v            |    MCP client     |
  +---------+  tool |  (this notebook)  |         MCP server 2: /api/2.0/mcp/genie/<space 2>  --> Genie space 2
  |   LLM   |------>+---------+---------+                 ^
  | (router)| calls           |                           |
  +---------+                 +---------------------------+
       ^                          Genie writes the SQL, runs it, returns the answer and the rows
       |  results
       +---- final answer, saying which space each number came from
```

**What it shows**
1. Each Genie space is an MCP server. We list its tools with the standard MCP `tools/list` call.
2. The LLM sees those tools, one per space, with the space's title and description. It **chooses** which to call:
   one space, both, or neither.
3. The MCP client runs the call (`tools/call`), waits for Genie to finish, and hands the result back to the LLM.
   The LLM then writes the answer.

**Nothing is deployed.** No model is logged or served, and no endpoint, app, job, cluster setting or package
install is needed. The notebook only *uses* what already exists:

| It uses | Already in your workspace |
|---|---|
| Your two Genie spaces | through their **managed MCP servers**, which Databricks runs for every space |
| An LLM | a **Foundation Model API** pay-per-token chat endpoint, such as `databricks-claude-sonnet-4-5` |
| Your identity | the notebook's own login; Genie runs every query **as you**, with your permissions |

The only trace it leaves: each question you send to a Genie space shows up in that space's conversation history,
exactly as if you had typed it in the Genie UI.

**Before you run**
* You can open both Genie spaces (*CAN RUN* or higher) and use the SQL warehouse each one is attached to.
* Your workspace has Foundation Model APIs. Any chat endpoint that supports tool calling works.
* Any compute: serverless or a cluster with Databricks Runtime 15.4 LTS or newer. Nothing to install.

**How to run:** paste the two Genie space IDs into the widgets at the top, then **Run all**.
You find the ID in the space's URL: `https://<workspace>/genie/rooms/`**`01f0...`**`?o=...`.
You can also paste the whole URL.

## 1 · Settings

Run this cell once. It creates the widgets at the top of the notebook. Then fill them in:

| Widget | What to enter |
|---|---|
| `genie_space_id_1`, `genie_space_id_2` | **required**: the IDs (or URLs) of your two Genie spaces |
| `genie_space_hint_1`, `genie_space_hint_2` | *optional*: one line on what each space covers. The LLM routes on the space's title and description; add a hint if those are thin |
| `llm_endpoint` | *optional*: a chat endpoint name. Empty = the notebook picks one (Claude first) |

Prefer typing them in code? Put them in `DEFAULTS` below instead: a value there fills the widget.
Outside Databricks the settings come from environment variables in upper case (`GENIE_SPACE_ID_1`, ...).

In [ ]:
import os

try:
    dbutils  # noqa: B018  (defined in every Databricks notebook)
    IN_DATABRICKS = True
except NameError:
    IN_DATABRICKS = False

DEFAULTS = {
    "genie_space_id_1": "",        # e.g. "01f0a1b2c3d4e5f6a7b8c9d0e1f2a3b4" or the space's URL
    "genie_space_id_2": "",
    "genie_space_hint_1": "",
    "genie_space_hint_2": "",
    "llm_endpoint": "",
}

WIDGETS = [
    ("genie_space_id_1", "Genie space 1: ID or URL"),
    ("genie_space_id_2", "Genie space 2: ID or URL"),
    ("genie_space_hint_1", "Space 1: what it covers (optional)"),
    ("genie_space_hint_2", "Space 2: what it covers (optional)"),
    ("llm_endpoint", "LLM endpoint (optional)"),
]
if IN_DATABRICKS:
    for name, label in WIDGETS:
        dbutils.widgets.text(name, DEFAULTS[name], label)


def setting(name: str) -> str:
    """A widget value in Databricks, an environment variable anywhere else; DEFAULTS when empty."""
    value = dbutils.widgets.get(name) if IN_DATABRICKS else os.environ.get(name.upper(), "")
    return value.strip() or DEFAULTS[name].strip()


print("Settings read from", "the widgets" if IN_DATABRICKS else "environment variables")

## 2 · Connect to the workspace and read the settings

`WorkspaceClient()` signs in as you, with no token to paste. It is part of the Databricks SDK, which every
Databricks runtime includes.

In [ ]:
import json
import re
import time

import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
ME = w.current_user.me().user_name


def space_id_from(value: str, widget: str) -> str:
    """Accept a bare space ID or any URL that contains one (Genie UI link, MCP URL)."""
    value = value.strip()
    match = re.search(r"/(?:genie/rooms|mcp/genie)/([0-9A-Za-z]+)", value)
    if match:
        return match.group(1)
    if re.fullmatch(r"[0-9A-Za-z]+", value):
        return value
    raise ValueError(f"Widget {widget}: '{value}' is not a Genie space ID or URL. Copy it from the Genie URL: "
                     f"https://<workspace>/genie/rooms/<space_id>")


missing = [n for n in ("genie_space_id_1", "genie_space_id_2") if not setting(n)]
if missing:
    raise ValueError(f"Fill in the widget(s) {', '.join(missing)} at the top of the notebook (or DEFAULTS in "
                     f"section 1) with your Genie space IDs, from the URL https://<workspace>/genie/rooms/<space_id>, "
                     f"then run again.")

SPACE_IDS = {
    "space_1": space_id_from(setting("genie_space_id_1"), "genie_space_id_1"),
    "space_2": space_id_from(setting("genie_space_id_2"), "genie_space_id_2"),
}
HINTS = {"space_1": setting("genie_space_hint_1"), "space_2": setting("genie_space_hint_2")}
if SPACE_IDS["space_1"] == SPACE_IDS["space_2"]:
    raise ValueError("Both widgets hold the same Genie space ID. Enter two different spaces.")

print(f"Workspace : {HOST}")
print(f"Signed in : {ME}")
for label, space_id in SPACE_IDS.items():
    print(f"{label}   : {space_id}  ->  MCP server {HOST}/api/2.0/mcp/genie/{space_id}")

## 3 · An MCP client in 60 lines

MCP (Model Context Protocol) is a small, open protocol: **JSON-RPC 2.0 messages over HTTPS**. Databricks
managed MCP servers use its *Streamable HTTP* transport. A client needs four messages:

| Message | What it does |
|---|---|
| `initialize` | agree on the protocol version; the server may return a session ID (`Mcp-Session-Id` header) |
| `notifications/initialized` | tell the server the client is ready |
| `tools/list` | which tools does the server offer, and which arguments do they take (JSON Schema)? |
| `tools/call` | run one tool with arguments |

The server answers either with plain JSON or with a short server-sent-event stream. The client below handles
both. It is the same protocol that Claude Desktop, Cursor or the `mcp` Python SDK speak. We write it out here so
you can see every message, and so the notebook needs no `%pip install`.

Two details matter on Databricks:
* **Authentication**: every request carries your Databricks token, taken from `w.config.authenticate()`.
* **Timeout**: a Genie tool call can keep the request open while Genie works, often 10 to 30 seconds. The client
  waits up to 5 minutes. Generic MCP clients default to a few seconds, which breaks here.

In [ ]:
MCP_LOG: list[tuple[str, str, dict]] = []      # (server, direction, message): every JSON-RPC message, for section 9


class MCPError(RuntimeError):
    pass


class MCPClient:
    """A minimal MCP client: JSON-RPC 2.0 over the Streamable HTTP transport."""

    PROTOCOL_VERSION = "2025-06-18"

    def __init__(self, url: str, name: str, timeout: float = 300):
        self.url, self.name, self.timeout = url, name, timeout
        self.session_id = None
        self.protocol_version = None
        self.server_info: dict = {}
        self._next_id = 0
        self._http = requests.Session()

    def connect(self) -> "MCPClient":
        self.session_id = self.protocol_version = None
        result = self._send("initialize", {"protocolVersion": self.PROTOCOL_VERSION, "capabilities": {},
                                           "clientInfo": {"name": "genie-mcp-router-notebook", "version": "1.0"}})
        self.protocol_version = result.get("protocolVersion", self.PROTOCOL_VERSION)
        self.server_info = result.get("serverInfo") or {}
        self._send("notifications/initialized", notification=True)
        return self

    def list_tools(self) -> list[dict]:
        tools, cursor = [], None
        while True:
            result = self.request("tools/list", {"cursor": cursor} if cursor else {})
            tools += result.get("tools", [])
            cursor = result.get("nextCursor")
            if not cursor:
                return tools

    def call_tool(self, name: str, arguments: dict) -> dict:
        return self.request("tools/call", {"name": name, "arguments": arguments})

    def request(self, method: str, params: dict) -> dict:
        try:
            return self._send(method, params)
        except MCPSessionExpired:                  # the server forgot our session: open a new one, retry once
            self.connect()
            return self._send(method, params)

    def _headers(self) -> dict:
        headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
        headers.update(w.config.authenticate())    # a fresh Authorization header on every request
        if self.session_id:
            headers["Mcp-Session-Id"] = self.session_id
        if self.protocol_version:
            headers["MCP-Protocol-Version"] = self.protocol_version
        return headers

    def _send(self, method: str, params: dict | None = None, notification: bool = False) -> dict | None:
        message = {"jsonrpc": "2.0", "method": method, "params": params or {}}
        if not notification:
            self._next_id += 1
            message["id"] = self._next_id
        MCP_LOG.append((self.name, "->", message))
        with self._http.post(self.url, json=message, headers=self._headers(), stream=True,
                             timeout=(30, self.timeout)) as response:
            if response.status_code == 404 and self.session_id and method != "initialize":
                raise MCPSessionExpired()
            if response.status_code >= 400:
                raise MCPError(f"{self.name}: HTTP {response.status_code} on {method}: {response.text[:500]}")
            self.session_id = response.headers.get("Mcp-Session-Id", self.session_id)
            if notification:
                return None
            response.encoding = "utf-8"
            if "text/event-stream" in response.headers.get("Content-Type", ""):
                replies = _sse_messages(response.iter_lines(decode_unicode=True))
            else:
                body = response.json()
                replies = body if isinstance(body, list) else [body]
            for reply in replies:
                if isinstance(reply, dict) and reply.get("id") == message["id"]:
                    MCP_LOG.append((self.name, "<-", reply))
                    if "error" in reply:
                        err = reply["error"]
                        raise MCPError(f"{self.name}: {method} failed: {err.get('message')} (code {err.get('code')})")
                    return reply.get("result") or {}
        raise MCPError(f"{self.name}: no reply to {method}")


class MCPSessionExpired(MCPError):
    pass


def _sse_messages(lines):
    """Yield the JSON messages of a server-sent-event stream (one per event's data lines)."""
    data = []
    for line in lines:
        if line is None:
            continue
        if line == "":
            if data:
                yield json.loads("\n".join(data))
                data = []
        elif line.startswith("data:"):
            data.append(line[5:].lstrip(" "))
    if data:
        yield json.loads("\n".join(data))


print("MCP client ready")

## 4 · Open both Genie MCP servers and list their tools

For each space the notebook:
* reads the space's **title and description** with the Genie REST API. These, plus your optional hint, are what
  the LLM routes on. Sample questions are read too, when your permission on the space allows it;
* connects to the space's **MCP server** and calls `tools/list`. A Genie space offers two tools:
  `query_space_<id>` asks a question and `poll_response_<id>` fetches the answer once Genie has finished.

In [ ]:
import pandas as pd

TERMINAL = {"COMPLETED", "SUCCEEDED", "FAILED", "ERROR", "CANCELLED", "CANCELED", "QUERY_RESULT_EXPIRED",
            "INCOMPLETE"}


def show(df: pd.DataFrame) -> None:
    """display() in a notebook, plain text elsewhere."""
    try:
        display(df)  # noqa: F821  (built in to Databricks and Jupyter notebooks)
    except NameError:
        print(df.to_string(index=False) if len(df) else "(no rows)")


def _camel(name: str) -> str:
    head, *rest = name.split("_")
    return head + "".join(p.title() for p in rest)


def _payload(result: dict):
    """The content of an MCP tool result: structured content if the server sent it, else the text (JSON or not)."""
    text = "\n".join(c.get("text", "") for c in result.get("content") or [] if c.get("type") == "text")
    if result.get("isError"):
        raise MCPError(text or "the tool reported an error")
    if result.get("structuredContent") is not None:
        return result["structuredContent"]
    try:
        return json.loads(text)
    except ValueError:
        return text


def _row(raw) -> list:
    if isinstance(raw, dict) and "values" in raw:              # {"values": [{"string_value": "..."}, ...]}
        return [next(iter(v.values()), None) if isinstance(v, dict) else v for v in raw["values"]]
    return list(raw) if isinstance(raw, (list, tuple)) else [raw]


class GenieAnswer:
    """What one Genie message returned: status, text, and for each query the SQL and the rows."""

    def __init__(self, space: "GenieSpace", payload, seconds: float, polls: int):
        self.space, self.seconds, self.polls = space, seconds, polls
        self.raw = payload
        self.texts, self.queries, self.error, self.conversation_id = [], [], None, None
        if not isinstance(payload, dict):
            self.status, self.texts = "COMPLETED", [str(payload)]
            return
        self.status = str(payload.get("status") or "COMPLETED").upper()
        self.conversation_id = payload.get("conversationId") or payload.get("conversation_id")
        content = payload.get("content") or {}
        if isinstance(content, str):
            content = {"textAttachments": [content]}
        for t in content.get("textAttachments") or []:
            self.texts.append(t if isinstance(t, str) else (t.get("content") or t.get("text") or json.dumps(t)))
        for q in content.get("queryAttachments") or []:
            resp = q.get("statement_response") or q.get("statementResponse") or {}
            manifest = resp.get("manifest") or {}
            columns = [c.get("name") for c in (manifest.get("schema") or {}).get("columns") or []]
            rows = [_row(r) for r in (resp.get("result") or {}).get("data_array") or []]
            self.queries.append({"sql": q.get("query"), "description": q.get("description"), "columns": columns,
                                 "rows": rows, "total_row_count": manifest.get("total_row_count", len(rows))})
        self.error = payload.get("error")
        if not self.texts and not self.queries and not self.error:     # a shape we do not know: keep it readable
            self.texts = [json.dumps(payload)[:4000]]

    def for_llm(self, max_rows: int = 30) -> str:
        """The result as compact text for the LLM."""
        lines = [f"Genie space: {self.space.title} | status: {self.status}"]
        if self.error:
            lines.append(f"Error: {self.error}")
        if self.texts:
            lines.append("Answer: " + " ".join(self.texts))
        for i, q in enumerate(self.queries, 1):
            lines.append(f"Query {i}: {q['description'] or ''}\nSQL: {q['sql']}")
            if q["columns"]:
                lines.append(f"Result ({q['total_row_count']} rows, first {min(max_rows, len(q['rows']))} shown):")
                lines.append(" | ".join(map(str, q["columns"])))
                lines += [" | ".join("NULL" if v is None else str(v) for v in r) for r in q["rows"][:max_rows]]
        return "\n".join(lines)

    def link(self) -> str | None:
        """The conversation in the Genie UI: Genie keeps every question asked through MCP in its history."""
        if not self.conversation_id:
            return None
        return f"{HOST}/genie/rooms/{self.space.space_id}/chats/{self.conversation_id}"

    def tables(self) -> list[pd.DataFrame]:
        return [pd.DataFrame(q["rows"], columns=q["columns"] or None) for q in self.queries if q["columns"]]


class GenieSpace:
    """One Genie space, reached through its managed MCP server."""

    def __init__(self, label: str, space_id: str, hint: str = ""):
        self.label, self.space_id, self.hint = label, space_id, hint
        self.title, self.description, self.sample_questions = space_id, "", []
        self._read_metadata()
        self.url = f"{HOST}/api/2.0/mcp/genie/{space_id}"
        self.mcp = MCPClient(self.url, name=label).connect()
        self.tools = self.mcp.list_tools()
        self.ask_tool = self._find_tool("query_space", ("query", "question"))
        self.poll_tool = self._find_tool("poll_response", ("message_id", "response_id"), required=False)

    def _read_metadata(self) -> None:
        path = f"/api/2.0/genie/spaces/{self.space_id}"
        try:
            space = w.api_client.do("GET", path, query={"include_serialized_space": "true"})
        except Exception:                          # reading the configuration needs CAN EDIT: fall back to the basics
            try:
                space = w.api_client.do("GET", path)
            except Exception as err:               # noqa: BLE001  (the MCP server is what matters; carry on)
                print(f"  note: could not read the details of {self.label} ({err}); routing uses the hint only")
                return
        self.title = space.get("title") or self.space_id
        self.description = space.get("description") or ""
        try:
            config = json.loads(space.get("serialized_space") or "{}").get("config") or {}
            for q in config.get("sample_questions") or []:
                text = q.get("question")
                text = " ".join(text) if isinstance(text, list) else text
                if text:
                    self.sample_questions.append(text)
        except (ValueError, AttributeError):
            pass

    def _find_tool(self, prefix: str, arg_names: tuple, required: bool = True) -> dict | None:
        for tool in self.tools:
            if tool["name"].startswith(prefix):
                return tool
        for tool in self.tools:                    # another naming: find the tool by its arguments
            if any(a in ((tool.get("inputSchema") or {}).get("properties") or {}) for a in arg_names):
                return tool
        if required:
            raise MCPError(f"{self.label}: no '{prefix}' tool among {[t['name'] for t in self.tools]}")
        return None

    def routing_description(self) -> str:
        """What the LLM reads to decide whether this space can answer a question."""
        parts = [f"Ask the Databricks Genie space '{self.title}' a business question in plain English. "
                 f"Genie writes and runs the SQL on this space's tables and returns the answer and the rows."]
        if self.description:
            parts.append(f"This space covers: {self.description}")
        if self.hint:
            parts.append(f"Note from the user: {self.hint}")
        if self.sample_questions:
            parts.append("Example questions: " + " | ".join(self.sample_questions[:5]))
        return "\n".join(parts)[:3000]

    def ask(self, arguments: dict, poll_every: float = 2, timeout: float = 600) -> GenieAnswer:
        """tools/call the ask tool, then poll until Genie has finished."""
        started, polls = time.time(), 0
        payload = _payload(self.mcp.call_tool(self.ask_tool["name"], arguments))
        ids = {}                                   # conversation and message IDs, from whichever reply carried them
        while self.poll_tool and isinstance(payload, dict) and str(payload.get("status") or "COMPLETED").upper() \
                not in TERMINAL:
            if time.time() - started > timeout:
                raise TimeoutError(f"{self.label}: Genie did not finish within {timeout:.0f} s")
            ids.update({k: v for k, v in payload.items() if v is not None})
            time.sleep(poll_every)
            payload = _payload(self.mcp.call_tool(self.poll_tool["name"], self._poll_arguments(ids)))
            polls += 1
        if isinstance(payload, dict) and ids:
            payload = {**{k: v for k, v in ids.items() if k != "status"}, **payload}
        return GenieAnswer(self, payload, time.time() - started, polls)

    def _poll_arguments(self, payload: dict) -> dict:
        schema = self.poll_tool.get("inputSchema") or {}
        args = {}
        for name in schema.get("properties") or {}:
            for key in (name, _camel(name)):
                if payload.get(key) is not None:
                    args[name] = payload[key]
                    break
        absent = [n for n in schema.get("required") or [] if n not in args]
        if absent:
            raise MCPError(f"{self.label}: the answer has no {absent} to poll with: {json.dumps(payload)[:300]}")
        return args


SPACES = []
for label, space_id in SPACE_IDS.items():
    print(f"Connecting to {label} ...")
    SPACES.append(GenieSpace(label, space_id, HINTS[label]))

show(pd.DataFrame([{
    "space": s.label, "title": s.title, "MCP server": s.mcp.server_info.get("name", ""),
    "protocol": s.mcp.protocol_version, "tools (tools/list)": ", ".join(t["name"] for t in s.tools),
    "sample questions": len(s.sample_questions),
} for s in SPACES]))

for s in SPACES:
    if not (s.description or s.hint or s.sample_questions):
        print(f"\nTip: {s.label} ('{s.title}') has no description. Put one line about its data in the widget "
              f"genie_space_hint_{s.label[-1]} so the LLM can route to it reliably.")

## 5 · The tools the LLM sees

The LLM gets **one tool per Genie space**: the space's `query_space_<id>` tool, with the argument schema the MCP
server returned in `tools/list`. Its description is the space's title, description, your hint and sample
questions. **The LLM routes on these descriptions**, so better descriptions mean better routing.

The poll tool is not given to the LLM. Waiting for Genie is the client's job, so the LLM only makes the decision
that matters: *which space?*

In [ ]:
names = [s.ask_tool["name"] for s in SPACES]
TOOLS, SPACE_BY_TOOL = [], {}
for s in SPACES:
    name = s.ask_tool["name"] if len(set(names)) == len(names) else f"{s.label}_{s.ask_tool['name']}"
    name = re.sub(r"[^A-Za-z0-9_-]", "_", name)[:64]
    schema = dict(s.ask_tool.get("inputSchema") or {"type": "object", "properties": {}})
    schema.setdefault("type", "object")
    schema.setdefault("properties", {})
    TOOLS.append({"type": "function", "function": {"name": name, "description": s.routing_description(),
                                                    "parameters": schema}})
    SPACE_BY_TOOL[name] = s

for tool in TOOLS:
    f = tool["function"]
    print(f"TOOL {f['name']}   (-> {SPACE_BY_TOOL[f['name']].label})")
    print("  arguments  :", json.dumps(f["parameters"].get("properties")))
    print("  description:", f["description"].replace("\n", "\n               "), "\n")

## 6 · Pick the LLM

The LLM is a Foundation Model API endpoint that already exists in your workspace (pay-per-token, nothing to
deploy). With the `llm_endpoint` widget empty, the notebook picks a chat endpoint: Claude first, then Llama,
GPT or Gemini. It sends one test message to check you can use it.

In [ ]:
PREFERENCE = [r"databricks-claude-sonnet", r"databricks-claude-opus", r"databricks-claude", r"databricks-gpt-5",
              r"databricks-meta-llama-3-3-70b", r"databricks-llama-4", r"databricks-gemini", r"databricks-gpt-oss",
              r"databricks-"]


def _ready(endpoint) -> bool:
    ready = getattr(getattr(endpoint, "state", None), "ready", None)
    return ready is None or str(getattr(ready, "value", ready)).upper() == "READY"


def pick_llm_endpoint() -> str:
    chosen = setting("llm_endpoint")
    if chosen:
        return chosen
    chat = sorted((e.name for e in w.serving_endpoints.list()
                   if "chat" in str(getattr(e, "task", "") or "") and _ready(e)), reverse=True)
    for pattern in PREFERENCE:
        for name in chat:                          # newest version first: names sort that way within a family
            if re.match(pattern, name):
                return name
    if chat:
        return chat[0]
    raise RuntimeError("No chat endpoint found. Enter one in the llm_endpoint widget (Serving > Endpoints lists "
                       "them), or ask an admin to enable Foundation Model APIs.")


def chat(messages: list[dict], tools: list[dict] | None = None, max_tokens: int = 1500) -> dict:
    """One chat completion from the serving endpoint (OpenAI-compatible request and response)."""
    body = {"messages": messages, "max_tokens": max_tokens}
    if tools:
        body["tools"] = tools
    return w.api_client.do("POST", f"/serving-endpoints/{LLM_ENDPOINT}/invocations", body=body)


def text_of(content) -> str:
    """Message content may be a string or a list of content blocks."""
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")
    return content or ""


LLM_ENDPOINT = pick_llm_endpoint()
reply = chat([{"role": "user", "content": "Reply with the single word OK."}], max_tokens=20)
print(f"LLM endpoint: {LLM_ENDPOINT}  (test reply: {text_of(reply['choices'][0]['message'].get('content')).strip()!r})")

## 7 · The router: the LLM decides, MCP does the work

`ask(question)` is the whole agent loop:
1. send the question and the tools to the LLM;
2. if it calls a tool, find the space behind it and run the call on that space's MCP server (`tools/call`,
   then poll until Genie has finished);
3. give the results back to the LLM and repeat, until it answers in plain text.

The LLM may call one space, **both** (for a question that needs data from each), or **none** (for a question
neither space can answer).

In [ ]:
SYSTEM_PROMPT = f"""You answer business questions with data from {len(SPACES)} Databricks Genie spaces.
Each tool asks one Genie space a question; read the tool descriptions to see which data each space holds.
- Call the tool of the space that holds the data the question needs. Pass the question in plain English, not SQL.
- If the question needs data from more than one space, call each of those tools, with the part of the question it can answer.
- If no space holds the data, say so and do not call a tool.
When you have the results, answer briefly, using only numbers the tools returned, and name the space each number came from."""

MAX_ROUNDS = 4


def ask(question: str, show_tables: bool = True) -> dict:
    print(f"QUESTION  {question}")
    started = time.time()
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    routed, answer = [], ""
    for round_no in range(1, MAX_ROUNDS + 1):
        if round_no == MAX_ROUNDS:
            messages.append({"role": "user", "content": "Answer now from the results above. Do not call tools."})
        message = chat(messages, tools=TOOLS)["choices"][0]["message"]
        calls = message.get("tool_calls") or []
        if not calls or round_no == MAX_ROUNDS:
            answer = text_of(message.get("content")).strip() or "(the LLM gave no text answer)"
            break
        for call in calls:                         # endpoints differ: keep arguments a JSON string
            if not isinstance(call["function"].get("arguments"), str):
                call["function"]["arguments"] = json.dumps(call["function"].get("arguments") or {})
        messages.append({"role": "assistant", "content": text_of(message.get("content")) or None,
                         "tool_calls": calls})
        for call in calls:
            name = call["function"]["name"]
            try:
                arguments = json.loads(call["function"]["arguments"] or "{}")
            except ValueError:
                arguments = {}
            space = SPACE_BY_TOOL.get(name)
            if space is None:
                result = f"There is no tool named {name}."
            else:
                print(f"  LLM -> {space.label} '{space.title}'  via MCP tools/call {name}")
                print(f"         arguments: {json.dumps(arguments)}")
                try:
                    genie = space.ask(arguments)
                    print(f"  MCP <- {genie.status} in {genie.seconds:.1f} s"
                          f" ({genie.polls} x {space.poll_tool['name'] if space.poll_tool else 'poll'})")
                    for q in genie.queries:
                        print("         SQL: " + (q["sql"] or "").strip().replace("\n", "\n              "))
                    if genie.link():
                        print(f"         open in Genie: {genie.link()}")
                    if show_tables:
                        for table in genie.tables():
                            show(table.head(20))
                    result = genie.for_llm()
                    routed.append(space.label)
                except Exception as err:           # noqa: BLE001  (tell the LLM, keep the demo going)
                    print(f"  MCP <- ERROR {err}")
                    result = f"The Genie space returned an error: {err}"
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    if not routed:
        print("  LLM -> no Genie space called")
    print(f"ANSWER    {answer}\n          ({time.time() - started:.1f} s)\n")
    return {"question": question, "routed to": ", ".join(dict.fromkeys(routed)) or "none", "answer": answer}


print("ask() is ready")

## 8 · Demo: watch the LLM route

Each demo question has an **expected** space, so the scorecard at the end shows whether the LLM routed it right.
The questions are your spaces' own sample questions. For a space without one, or whose sample questions you
cannot read, the LLM writes a question from the space's description. To use your own questions, fill in
`MY_QUESTIONS`.

There are also two special questions: one that **needs both spaces**, and one that **neither** space can answer.

In [ ]:
MY_QUESTIONS = [
    # ("What were net sales by region last week?", "space_1"),     # (question, the space you expect)
]


def write_question(prompt: str) -> str:
    reply = chat([{"role": "user", "content": prompt}], max_tokens=200)
    text = text_of(reply["choices"][0]["message"].get("content")).strip().strip('"').splitlines()
    return text[0].strip() if text else ""


def describe(s: GenieSpace) -> str:
    return f"'{s.title}': {s.description or '(no description)'} {s.hint}".strip()


QUESTIONS = list(MY_QUESTIONS)
if not QUESTIONS:
    for s in SPACES:
        question = s.sample_questions[0] if s.sample_questions else write_question(
            f"Here is a Databricks Genie space (a data room business users ask questions to) {describe(s)}\n"
            f"Write ONE short, concrete business question this space can answer. Reply with the question only.")
        if question:
            QUESTIONS.append((question, s.label))
    both = write_question(
        "Here are two Databricks Genie spaces:\n" + "\n".join(f"- {describe(s)}" for s in SPACES) +
        "\nWrite ONE short business question that needs data from BOTH spaces to answer. "
        "Reply with the question only.")
    if both:
        QUESTIONS.append((both, "space_1, space_2"))
    QUESTIONS.append(("What is the capital of France?", "none"))

for q, expected in QUESTIONS:
    print(f"- {q}   [expected: {expected}]")

In [ ]:
RESULTS = []
for question, expected in QUESTIONS:
    result = ask(question)
    result["expected"] = expected
    RESULTS.append(result)


def _same(a: str, b: str) -> bool:
    return sorted(x.strip() for x in a.split(",")) == sorted(x.strip() for x in b.split(","))


scorecard = pd.DataFrame([{"question": r["question"], "expected": r["expected"], "routed to": r["routed to"],
                           "routed right": "yes" if _same(r["expected"], r["routed to"]) else "NO"}
                          for r in RESULTS])
print("Routing scorecard")
show(scorecard)
print(f"\n{(scorecard['routed right'] == 'yes').sum()} of {len(scorecard)} questions routed as expected")

## 9 · Under the hood: the MCP messages

Every message the client sent and received is in `MCP_LOG`. Here are the messages of one Genie question: the
`tools/call` that asked it, and the reply that carried the answer. This is all MCP is on the wire.

In [ ]:
def short(message: dict, limit: int = 1500) -> str:
    text = json.dumps(message, indent=2)
    return text if len(text) <= limit else text[:limit] + f"\n  ... ({len(text) - limit} more characters)"


calls = [i for i, (_, direction, m) in enumerate(MCP_LOG) if direction == "->" and m.get("method") == "tools/call"]
if calls:
    server, _, request = MCP_LOG[calls[0]]
    replies = [m for name, direction, m in MCP_LOG[calls[0]:] if name == server and direction == "<-"]
    print(f"--- {server}: tools/call request ---\n{short(request)}\n")
    print(f"--- {server}: its reply ---\n{short(replies[0])}\n")
    done = next((m for m in replies if "COMPLETED" in json.dumps(m)), None)
    if done is not None and done is not replies[0]:
        print(f"--- {server}: the reply that carried the answer (after polling) ---\n{short(done)}\n")
else:
    print("No tool was called yet: run section 8 first.")

counts = pd.Series([m.get("method", "reply") for _, d, m in MCP_LOG if d == "->"]).value_counts()
print("\nMessages sent in this session:", ", ".join(f"{k}: {v}" for k, v in counts.items()))

## 10 · Your turn

Type any question. The LLM picks the space, or both, or neither.

In [ ]:
MY_QUESTION = ""          # e.g. "Which region grew fastest this quarter?"

if MY_QUESTION.strip():
    ask(MY_QUESTION)
else:
    print("Set MY_QUESTION above and run this cell again.")

## 11 · What you just saw, and what was left behind

* **MCP made the Genie spaces tools.** The notebook found each space's tools with `tools/list` and ran them with
  `tools/call`. It needed no Genie-specific SDK, only the protocol.
* **The LLM did the routing.** It read the tool descriptions and chose a space, both, or none for each question.
  Better space descriptions (or hints) mean better routing.
* **Genie stayed in charge of the data.** It wrote and ran the SQL as you, with your Unity Catalog permissions.

**Nothing was deployed or changed** in the workspace. The only traces are the conversations in each space's
Genie history, and the LLM's pay-per-token usage.

**Taking it further** (optional, not needed for this demo)
* Add more Genie spaces: one more `GenieSpace(...)` in section 4. The LLM simply gets one more tool.
* Mix in other managed MCP servers, such as Unity Catalog functions (`/api/2.0/mcp/functions/<catalog>/<schema>`)
  or Vector Search (`/api/2.0/mcp/vector-search/<catalog>/<schema>`). Same client, same loop.
* To serve this to other people, wrap the loop as an agent (the `databricks-mcp` package and the Agent Framework).
  That *is* a deployment, and the reason this notebook stops here.